# NautiCost — Cost Prediction Modeling

Predict `final_charge` per yacht service transaction.

**Approach:** LightGBM (primary) + Ridge baseline, time-based split, log-transformed target.
See `../../.claude/plans/staged-swimming-treehouse.md` for the full plan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

import lightgbm as lgb

pd.set_option('display.max_columns', 50)
RNG = 42

## 1. Load data

In [ ]:
DATA = Path('../004 data/costs_merged.csv')
df = pd.read_csv(DATA)
print(df.shape)
df.head()

In [ ]:
# Drop rows with missing target
df = df.dropna(subset=['final_charge'])
df = df[df['final_charge'] > 0].copy()
print('After filtering:', df.shape)
df['final_charge'].describe()

## 2. Features and target

In [ ]:
CAT_FEATURES = ['office', 'arrival_port', 'service_type', 'service_category', 'size_category', 'loskrav']
NUM_FEATURES = ['gt', 'loa_m', 'beam_m', 'draft_m', 'fuel_lph', 'stay_days', 'month']
FEATURES = CAT_FEATURES + NUM_FEATURES
TARGET = 'final_charge'

for c in CAT_FEATURES:
    df[c] = df[c].astype('category')

df[FEATURES].dtypes

### 2b. Feature engineering
Temporal, interaction, and text-based features derived from existing columns.

In [ ]:
# ---- Temporal ----
df['quarter'] = ((df['month'] - 1) // 3 + 1).astype('int8')
df['is_summer'] = df['month'].isin([6, 7, 8]).astype('int8')
df['is_shoulder'] = df['month'].isin([5, 9]).astype('int8')

# ---- Interactions (numeric) ----
df['gt_x_stay'] = df['gt'].fillna(0) * df['stay_days'].fillna(0)
df['loa_x_stay'] = df['loa_m'].fillna(0) * df['stay_days'].fillna(0)
df['fuel_x_stay'] = df['fuel_lph'].fillna(0) * df['stay_days'].fillna(0)

# ---- Yacht-level aggregates (computed on train period only to avoid leakage) ----
hist = df[df['year'] <= 2024]
yacht_stats = hist.groupby('yacht_id')['final_charge'].agg(
    yacht_mean_charge='mean', yacht_visit_count='count'
).reset_index()
df = df.merge(yacht_stats, on='yacht_id', how='left')
df['yacht_mean_charge'] = df['yacht_mean_charge'].fillna(df['yacht_mean_charge'].median())
df['yacht_visit_count'] = df['yacht_visit_count'].fillna(0)

# ---- Text signal from invoice_comments ----
cmt = df['invoice_comments'].fillna('').astype(str).str.lower()
df['cmt_len'] = cmt.str.len().astype('int16')
df['cmt_has_urgent'] = cmt.str.contains('urgent|asap|rush', regex=True).astype('int8')
df['cmt_has_repair'] = cmt.str.contains('repair|fix|broken', regex=True).astype('int8')
df['cmt_has_fuel'] = cmt.str.contains('fuel|diesel|bunker', regex=True).astype('int8')

NEW_NUM = ['quarter','is_summer','is_shoulder','gt_x_stay','loa_x_stay','fuel_x_stay',
           'yacht_mean_charge','yacht_visit_count','cmt_len','cmt_has_urgent',
           'cmt_has_repair','cmt_has_fuel']
NUM_FEATURES = NUM_FEATURES + NEW_NUM
FEATURES = CAT_FEATURES + NUM_FEATURES
print(f'Total features: {len(FEATURES)} ({len(CAT_FEATURES)} cat + {len(NUM_FEATURES)} num)')

## 3. Time-based split
train ≤ 2024, val = 2025, test = 2026

In [ ]:
train = df[df['year'] <= 2024]
val   = df[df['year'] == 2025]
test  = df[df['year'] == 2026]
print('train', train.shape, 'val', val.shape, 'test', test.shape)

def xy(sub):
    X = sub[FEATURES].copy()
    y = np.log1p(sub[TARGET].values)
    return X, y

X_tr, y_tr = xy(train)
X_va, y_va = xy(val)
X_te, y_te = xy(test)

## 4. Baseline: median predictor & Ridge

In [ ]:
def eval_pred(y_true_log, y_pred_log, label):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f'{label:20s} MAE={mae:10.0f}  RMSE={rmse:10.0f}  MAPE={mape:6.1f}%')
    return mae, rmse, mape

med = np.median(y_tr)
eval_pred(y_va, np.full_like(y_va, med), 'Median baseline')

In [ ]:
pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_FEATURES),
    ('num', StandardScaler(), NUM_FEATURES),
])
ridge = Pipeline([('pre', pre), ('reg', Ridge(alpha=1.0, random_state=RNG))])
ridge.fit(X_tr.fillna(-1), y_tr)
eval_pred(y_va, ridge.predict(X_va.fillna(-1)), 'Ridge (val)')

## 5. LightGBM

In [ ]:
lgb_params = dict(
    objective='regression',
    metric='mae',
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=10,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    random_state=RNG,
    verbose=-1,
)

dtrain = lgb.Dataset(X_tr, y_tr, categorical_feature=CAT_FEATURES)
dval   = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtrain)

model = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)

In [ ]:
pred_va = model.predict(X_va, num_iteration=model.best_iteration)
eval_pred(y_va, pred_va, 'LightGBM (val)')

if len(X_te):
    pred_te = model.predict(X_te, num_iteration=model.best_iteration)
    eval_pred(y_te, pred_te, 'LightGBM (test)')

## 6. Feature importance

In [ ]:
imp = pd.DataFrame({
    'feature': model.feature_name(),
    'gain': model.feature_importance(importance_type='gain'),
}).sort_values('gain', ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(imp['feature'], imp['gain'])
ax.set_title('LightGBM feature importance (gain)')
plt.tight_layout()
plt.show()

## 7. Quantile predictions (P10 / P50 / P90)

In [ ]:
quantile_models = {}
for q in [0.1, 0.5, 0.9]:
    p = {**lgb_params, 'objective': 'quantile', 'alpha': q, 'metric': 'quantile'}
    m = lgb.train(p, dtrain, num_boost_round=model.best_iteration or 500,
                  valid_sets=[dval], callbacks=[lgb.log_evaluation(0)])
    quantile_models[q] = m

preds = {q: np.expm1(m.predict(X_va)) for q, m in quantile_models.items()}
out = pd.DataFrame({
    'actual': np.expm1(y_va),
    'p10': preds[0.1], 'p50': preds[0.5], 'p90': preds[0.9],
}).head(15)
out

## 8. CatBoost (native categorical handling)

In [ ]:
from catboost import CatBoostRegressor

X_tr_cb = X_tr.copy()
X_va_cb = X_va.copy()
for c in CAT_FEATURES:
    X_tr_cb[c] = X_tr_cb[c].astype(str).fillna('NA')
    X_va_cb[c] = X_va_cb[c].astype(str).fillna('NA')

cb = CatBoostRegressor(
    iterations=2000, learning_rate=0.05, depth=6,
    loss_function='MAE', cat_features=CAT_FEATURES,
    early_stopping_rounds=50, random_seed=RNG, verbose=0,
)
cb.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va))
eval_pred(y_va, cb.predict(X_va_cb), 'CatBoost (val)')

## 9. Hyperparameter tuning with Optuna (LightGBM)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    p = dict(
        objective='regression', metric='mae', verbose=-1, random_state=RNG,
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 127),
        min_data_in_leaf=trial.suggest_int('min_data_in_leaf', 5, 50),
        feature_fraction=trial.suggest_float('feature_fraction', 0.6, 1.0),
        bagging_fraction=trial.suggest_float('bagging_fraction', 0.6, 1.0),
        bagging_freq=5,
        lambda_l2=trial.suggest_float('lambda_l2', 1e-3, 10, log=True),
    )
    dtr = lgb.Dataset(X_tr, y_tr, categorical_feature=CAT_FEATURES)
    dva = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtr)
    m = lgb.train(p, dtr, num_boost_round=1500, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    pred = m.predict(X_va, num_iteration=m.best_iteration)
    return mean_absolute_error(np.expm1(y_va), np.expm1(pred))

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, show_progress_bar=False)
print('Best MAE:', study.best_value)
print('Best params:', study.best_params)

In [ ]:
# Train final tuned LightGBM model
tuned_params = dict(objective='regression', metric='mae', verbose=-1,
                    random_state=RNG, bagging_freq=5, **study.best_params)
dtr = lgb.Dataset(X_tr, y_tr, categorical_feature=CAT_FEATURES)
dva = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtr)
best_model = lgb.train(
    tuned_params, dtr, num_boost_round=2000, valid_sets=[dva],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
)
eval_pred(y_va, best_model.predict(X_va, num_iteration=best_model.best_iteration), 'Tuned LightGBM')

## 10. SHAP explainability

In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_va)

# Summary plot (global feature impact)
shap.summary_plot(shap_values, X_va, plot_type='bar', show=False)
plt.tight_layout(); plt.show()

shap.summary_plot(shap_values, X_va, show=False)
plt.tight_layout(); plt.show()

In [ ]:
# Per-row explanation: top 5 rows with largest prediction
pred_va = best_model.predict(X_va, num_iteration=best_model.best_iteration)
top_idx = np.argsort(-pred_va)[:5]
for i in top_idx:
    print(f"row={i}  actual={np.expm1(y_va[i]):,.0f}  predicted={np.expm1(pred_va[i]):,.0f}")
    contribs = pd.Series(shap_values[i], index=FEATURES).sort_values(key=np.abs, ascending=False)
    print(contribs.head(5).to_string(), '\n')